<a href="https://colab.research.google.com/github/Innovatewithapple/Brick/blob/main/FourBricks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import tensorflow as tf
from tensorflow.keras import layers

Embedding + Position

In [3]:
vocab_size = 8
max_length = 10
embed_dim = 4

In [4]:
input_layer = layers.Embedding(input_dim=vocab_size,output_dim=embed_dim)

input = tf.constant([[1,2,3,4,5,6,7,2,4,7]])

connectlayer = input_layer(input)
connectlayer.shape

TensorShape([1, 10, 4])

In [5]:
position_layer = layers.Embedding(input_dim=connectlayer.shape[1],output_dim=embed_dim)
position_range = tf.range(start=0,limit=connectlayer.shape[1],delta=1)

position_output = position_layer(position_range)
position_output.shape

TensorShape([10, 4])

In [6]:
final_output = connectlayer + position_output
final_output.shape

TensorShape([1, 10, 4])

Self-Attention

In [7]:
self_attention = layers.MultiHeadAttention(num_heads=2,key_dim=embed_dim)

In [8]:
final_attention = self_attention(query=final_output,key=final_output,value=final_output)
final_attention.shape

TensorShape([1, 10, 4])

Brick3: MLP (Multi-Layer Perceptron)

In [9]:
expansion_dim = 16 # should be 4x of the features
expansion_layer = layers.Dense(expansion_dim,activation='relu')(final_attention)

In [10]:
expansion_layer_output = layers.Dense(4)(expansion_layer)
expansion_layer_output.shape

TensorShape([1, 10, 4])

Brick4: The Glue

In [13]:
# 1. Setup the Normalizer
# This acts as the "Volume Control" to keep the numbers stable.
norm = layers.LayerNormalization(epsilon=1e-6)

res_1= norm(final_output + final_attention)

# 3. GLUE #2: Add Glue #1 to MLP Output
# In your code: res1 + expansion_layer_output
final_block_output = norm(res_1 + expansion_layer_output)

print("Final Transformer Block Shape:", final_block_output.shape)

Final Transformer Block Shape: (1, 10, 4)


Classifier head

In [14]:
summary = layers.GlobalAveragePooling1D()(final_block_output)

prediction = layers.Dense(1,activation='sigmoid')(summary)

print("Final Prediction Shape:", prediction.shape)
print("The AI's Answer:", prediction.numpy())

Final Prediction Shape: (1, 1)
The AI's Answer: [[0.4861985]]
